# 03 — Model Metrics
ROC, Precision-Recall, Confusion Matrix e análise de thresholds.

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, confusion_matrix,
    classification_report, f1_score,
)
from sklearn.model_selection import train_test_split

from decision_service.models  import ChurnModel
from decision_service.dataset import ChurnDataset

MODEL_DIR = Path('../ml/models')
DATA_PATH = Path('../data/train_dataset.csv')

sns.set_theme(style='whitegrid')

In [ ]:
# Carregar modelo e scaler
churn_model  = joblib.load(MODEL_DIR / 'churn_model.pkl')
churn_scaler = joblib.load(MODEL_DIR / 'churn_scaler.pkl')

config   = ChurnModel()
dataset  = ChurnDataset(DATA_PATH, config)
X, y, _, _ = dataset.prepare()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = config.test_size,
    random_state = config.random_state,
    stratify     = y,
)

y_proba = churn_model.predict_proba(X_test)[:, 0]  # P(churnou)
y_pred  = churn_model.predict(X_test)

auc = roc_auc_score(y_test, y_proba)
print(f'AUC-ROC: {auc:.4f}')
print(f'Test size: {len(y_test)}')

## ROC Curve

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba, pos_label=0)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ChurnModel (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label='Baseline')
plt.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — ChurnModel')
plt.legend()
plt.tight_layout()
plt.show()

## Precision-Recall Curve

In [ ]:
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_proba, pos_label=0)
avg_prec = average_precision_score(y_test, y_proba, pos_label=0)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, color='#e74c3c', lw=2, label=f'Avg Precision = {avg_prec:.3f}')
baseline_p = (y_test == 0).mean()
plt.axhline(y=baseline_p, color='gray', linestyle='--', label=f'Baseline ({baseline_p:.2f})')
plt.fill_between(recall, precision, alpha=0.1, color='#e74c3c')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — ChurnModel')
plt.legend()
plt.tight_layout()
plt.show()

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Reds',
    xticklabels=['Prev. Churn', 'Prev. Renovação'],
    yticklabels=['Real Churn', 'Real Renovação'],
)
plt.title('Confusion Matrix — ChurnModel')
plt.ylabel('Verdadeiro')
plt.xlabel('Predito')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Churnou', 'Renovou']))

## Threshold Analysis
Qual threshold maximiza F1? Qual minimiza FN (churns não detectados)?

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
results = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)  # 1 = churn
    # Mapeamento inverso: proba alta = churnou (label 0)
    # Como y_proba = P(churnou), threshold alto = mais conservador
    tp = ((y_pred_t == 1) & (y_test == 0)).sum()
    fp = ((y_pred_t == 1) & (y_test == 1)).sum()
    fn = ((y_pred_t == 0) & (y_test == 0)).sum()
    tn = ((y_pred_t == 0) & (y_test == 1)).sum()
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    results.append({'threshold': t, 'precision': prec, 'recall': rec, 'f1': f1,
                    'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})

thresh_df = pd.DataFrame(results)

plt.figure(figsize=(10, 4))
plt.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision', color='#3498db')
plt.plot(thresh_df['threshold'], thresh_df['recall'],    label='Recall',    color='#e67e22')
plt.plot(thresh_df['threshold'], thresh_df['f1'],        label='F1',        color='#e74c3c', lw=2)
best_t = thresh_df.loc[thresh_df['f1'].idxmax(), 'threshold']
plt.axvline(x=best_t, linestyle='--', color='gray', label=f'Best F1 @ {best_t:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 por Threshold')
plt.legend()
plt.tight_layout()
plt.show()

print(f'\nMelhor threshold por F1: {best_t:.2f}')
print(thresh_df.round(3).to_string(index=False))

## Impacto Financeiro por Threshold

In [ ]:
df_full = pd.read_csv(DATA_PATH)
_, df_test, _, _ = train_test_split(
    df_full, y,
    test_size=config.test_size, random_state=config.random_state, stratify=y
)
df_test = df_test.reset_index(drop=True)

fin_results = []
for t in thresholds:
    flagged_churn   = (y_proba >= t)       # modelo alerta
    actual_churn    = (y_test  == 0)       # realmente churnou
    caught_mrr      = df_test.loc[flagged_churn & actual_churn, 'mrr'].sum()
    missed_mrr      = df_test.loc[~flagged_churn & actual_churn, 'mrr'].sum()
    false_alarm_mrr = df_test.loc[flagged_churn & ~actual_churn, 'mrr'].sum()
    fin_results.append({
        'threshold':      t,
        'mrr_recuperado': caught_mrr,
        'mrr_perdido':    missed_mrr,
        'mrr_falso_alarm': false_alarm_mrr,
    })

fin_df = pd.DataFrame(fin_results)

plt.figure(figsize=(10, 4))
plt.plot(fin_df['threshold'], fin_df['mrr_recuperado'] / 1000,  label='MRR Recuperado (k)',  color='#2ecc71')
plt.plot(fin_df['threshold'], fin_df['mrr_perdido'] / 1000,     label='MRR Perdido (k)',      color='#e74c3c')
plt.plot(fin_df['threshold'], fin_df['mrr_falso_alarm'] / 1000, label='Falso Alarme MRR (k)', color='#f39c12', linestyle='--')
plt.xlabel('Threshold')
plt.ylabel('MRR (R$ mil)')
plt.title('Impacto Financeiro por Threshold')
plt.legend()
plt.tight_layout()
plt.show()

print(fin_df.round(0).to_string(index=False))